<a href="https://colab.research.google.com/github/eiyu18/MachineLearning/blob/main/JS09/JS09_Assignment01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install annoy faiss-cpu hnswlib numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 7.8 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 26.8 MB/s  0:00:00
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520037 sha256=867cb637eea347b4f42499d9b915899c58571b7df7f60f08d3d6dece65f0585b
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679798 sha256=cf9c26be510f2a99a5ac8504e01888a55da12b796f1f487412035af9702e06d6
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50

In [2]:
import time
import numpy as np
import faiss
import hnswlib
from annoy import AnnoyIndex

K = 10
NQ = 100
rng = np.random.default_rng(42)

def normalize(a):
    return a / np.linalg.norm(a, axis=1, keepdims=True)

def ground_truth(X, Q, metric):
    if metric == "cosine":
        X, Q = normalize(X), normalize(Q)
        idx = faiss.IndexFlatIP(X.shape[1])
    else:
        idx = faiss.IndexFlatL2(X.shape[1])
    idx.add(X)
    return idx.search(Q, K)[1]

def recall(found, truth):
    return np.mean([len(set(f) & set(t)) / K for f, t in zip(found, truth)])

def run_annoy(X, Q, metric):
    d = X.shape[1]
    t = time.time()
    idx = AnnoyIndex(d, "angular" if metric == "cosine" else "euclidean")
    for i, v in enumerate(X):
        idx.add_item(i, v)
    idx.build(10)
    build = time.time() - t
    t = time.time()
    res = [idx.get_nns_by_vector(q, K) for q in Q]
    return build, (time.time() - t) / len(Q) * 1000, np.array(res)

def run_faiss(X, Q, metric):
    d = X.shape[1]
    if metric == "cosine":
        X, Q = normalize(X), normalize(Q)
        idx = faiss.IndexHNSWFlat(d, 16, faiss.METRIC_INNER_PRODUCT)
    else:
        idx = faiss.IndexHNSWFlat(d, 16)
    t = time.time()
    idx.add(X)
    build = time.time() - t
    t = time.time()
    res = idx.search(Q, K)[1]
    return build, (time.time() - t) / len(Q) * 1000, res

def run_hnsw(X, Q, metric):
    d = X.shape[1]
    idx = hnswlib.Index(space="cosine" if metric == "cosine" else "l2", dim=d)
    t = time.time()
    idx.init_index(max_elements=len(X), ef_construction=200, M=16)
    idx.add_items(X)
    build = time.time() - t
    idx.set_ef(50)
    t = time.time()
    res = idx.knn_query(Q, k=K)[0]
    return build, (time.time() - t) / len(Q) * 1000, res

rows = []
for n in (1_000, 1_000_000):
    for d in (2, 5):
        X = rng.random((n, d), dtype=np.float32)
        Q = rng.random((NQ, d), dtype=np.float32)
        for metric in ("euclidean", "cosine"):
            truth = ground_truth(X, Q, metric)
            row = [f"{n:,}/{d}D", metric]
            for fn in (run_annoy, run_faiss, run_hnsw):
                b, q, res = fn(X, Q, metric)
                row.append(f"build {b:.2f}s | query {q:.3f}ms | recall {recall(res, truth):.3f}")
            rows.append(row)
            print(" || ".join(row))

1,000/2D || euclidean || build 0.05s | query 0.034ms | recall 0.999 || build 0.06s | query 0.006ms | recall 1.000 || build 0.06s | query 0.011ms | recall 1.000
1,000/2D || cosine || build 0.04s | query 0.022ms | recall 0.999 || build 0.01s | query 0.005ms | recall 0.998 || build 0.04s | query 0.010ms | recall 0.999
1,000/5D || euclidean || build 0.02s | query 0.021ms | recall 0.967 || build 0.02s | query 0.007ms | recall 0.998 || build 0.06s | query 0.014ms | recall 1.000
1,000/5D || cosine || build 0.03s | query 0.023ms | recall 0.981 || build 0.02s | query 0.006ms | recall 1.000 || build 0.06s | query 0.015ms | recall 1.000
1,000,000/2D || euclidean || build 26.57s | query 0.057ms | recall 0.998 || build 26.29s | query 0.014ms | recall 1.000 || build 104.55s | query 0.039ms | recall 1.000
1,000,000/2D || cosine || build 88.98s | query 0.053ms | recall 0.075 || build 38.21s | query 0.018ms | recall 0.563 || build 139.14s | query 0.036ms | recall 0.104
1,000,000/5D || euclidean || buil